In [49]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
import json
import shutil
import os

In [4]:
DATA_DIR = Path("/Volumes/Extension0/Graduate/100.MAIN/01.DATA/1.Training/LABEL")
PHOTO_DIR = Path("/Volumes/Extension0/Graduate/100.MAIN/01.DATA/1.Training/PHOTO")
DATA_VAL_DIR = Path("/Volumes/Extension0/Graduate/100.MAIN/01.DATA/2.Validation/LABEL")
PHOTO_VAL_DIR = Path("/Volumes/Extension0/Graduate/100.MAIN/01.DATA/2.Validation/PHOTO")

DIR_REGEX = re.compile('[A-Z]+_3.상추[0-9]+')

REF_PHOTO_DIR = Path("/Users/mainframe/Workspace/Graduate/dataset/train/photos")
REF_LABEL_DIR = Path("/Users/mainframe/Workspace/Graduate/dataset/train/labels")
REF_PHOTO_VAL_DIR = Path("/Users/mainframe/Workspace/Graduate/dataset/validate/photos")
REF_LABEL_VAL_DIR = Path("/Users/mainframe/Workspace/Graduate/dataset/validate/labels")

MIN_DIR = 54
MAX_DIR = 75
MIN_VAL_DIR = 7
MAX_VAL_DIR = 10

In [5]:
def extract_photos_from_src_dir(src_dir, dst_dir, min_no, max_no):
    list_of_photos = []
    
    #Move Every Photos that has good portrait.
    for entry in src_dir.iterdir():
        if entry.is_dir() and DIR_REGEX.match(entry.name):
            dir_no = int(entry.name.split('상추')[1])
            if min_no <= dir_no and dir_no <= max_no:
                for subentry in entry.iterdir():
                    if subentry.is_file() and subentry.name != ".DS_Store":
                        list_of_photos.append(subentry)
                        shutil.copy(subentry, dst_dir / subentry.name)
    
    return list_of_photos

In [6]:
list_of_photos = extract_photos_from_src_dir(PHOTO_DIR, REF_PHOTO_DIR, MIN_DIR, MAX_DIR)

FileNotFoundError: [Errno 2] No such file or directory: '/Volumes/Extension0/Graduate/100.MAIN/01.DATA/1.Training/PHOTO'

In [7]:
list_of_photos_val = extract_photos_from_src_dir(PHOTO_VAL_DIR, REF_PHOTO_VAL_DIR, MIN_VAL_DIR, MAX_VAL_DIR)

FileNotFoundError: [Errno 2] No such file or directory: '/Volumes/Extension0/Graduate/100.MAIN/01.DATA/2.Validation/PHOTO'

In [55]:
def move_labels_from_src_dir(src_dir, dst_dir):
    #Perform and exhuastive search, and yes this is intentional.
    for entry in src_dir.iterdir():
        if entry.is_dir() and DIR_REGEX.match(entry.name):
            for subentry in entry.iterdir():
                if subentry.is_file() and subentry.name != ".DS_Store":
                    #print(f"Moving {subentry} to {REF_PHOTO_DIR / subentry.name}")
                    shutil.copy(subentry, dst_dir / subentry.name)

In [56]:
move_labels_from_src_dir(DATA_DIR, REF_LABEL_DIR)
move_labels_from_src_dir(DATA_VAL_DIR, REF_LABEL_VAL_DIR)

In [17]:
def get_from_dataset(src_dir):
    list_of_photos = []
    for entry in src_dir.iterdir():
        if entry.is_file() and entry.name != ".DS_Store":
            list_of_photos.append(entry)
    return list_of_photos

In [75]:
list_of_photos = get_from_dataset(REF_PHOTO_DIR)
list_of_photos_val = get_from_dataset(REF_PHOTO_VAL_DIR)

In [76]:
list_of_photos[0:5]

[PosixPath('/Users/mainframe/Workspace/Graduate/dataset/train/photos/C30_L01_06_003_756540.jpg'),
 PosixPath('/Users/mainframe/Workspace/Graduate/dataset/train/photos/C30_L01_06_003_806752.jpg'),
 PosixPath('/Users/mainframe/Workspace/Graduate/dataset/train/photos/C30_L01_02_002_492831.jpg'),
 PosixPath('/Users/mainframe/Workspace/Graduate/dataset/train/photos/C30_L01_06_001_778768.jpg'),
 PosixPath('/Users/mainframe/Workspace/Graduate/dataset/train/photos/C46_L01_02_006_571147.jpg')]

In [77]:
list_of_photos_val[0:5]

[PosixPath('/Users/mainframe/Workspace/Graduate/dataset/validate/photos/C30_L01_05_001_718559.jpg'),
 PosixPath('/Users/mainframe/Workspace/Graduate/dataset/validate/photos/C46_L01_02_004_641843.jpg'),
 PosixPath('/Users/mainframe/Workspace/Graduate/dataset/validate/photos/C30_L01_05_001_618571.jpg'),
 PosixPath('/Users/mainframe/Workspace/Graduate/dataset/validate/photos/C30_L01_06_002_756672.jpg'),
 PosixPath('/Users/mainframe/Workspace/Graduate/dataset/validate/photos/C31_L01_09_001_468379.jpg')]

In [78]:
def safe_avg_of(data, column_name):
    column = [e[column_name] for e in data]
    if None in column:
        return None
    num_column = [float(n) for n in column]
    return sum(num_column) / len(num_column)

In [79]:
def growth_to_number(growth):
    if growth == '정식기':
        return 1
    elif growth == '생육기':
        return 2
    elif growth == '수확기':
        return 3
    raise ValueError("Some other value has been detected")

In [80]:
def parse_json_file(file_path):
    """Reads a single JSON file and returns a dictionary or None if failed."""
    try:
        # Using pathlib's read_text handles the file opening/closing for you
        #dir_no, file_path = file_info
        raw_data = file_path.read_text(encoding='utf-8')
        
        data = json.loads(raw_data)

        leaf_ids = []
        stem_ids = []

        for cat in data["categories"]:
            if cat["name"] == "잎":
                leaf_ids.append(cat["id"])
            else:
                stem_ids.append(cat["id"])

        environment = data["envrionments"]
        
        extracted = {
            #'dir_no': dir_no,
            'file_name': file_path.name.split(".")[0],
            'crops_id': file_path.name[0:14],
            'growth': growth_to_number(data["images"]["growth_stage"]),
            'captured': data["images"]["date_captured"],
            'kind': data["images"]["kind_type"],
            'temp': safe_avg_of(environment, "ti_value"),
            'humid': safe_avg_of(environment, "hi_value"),
            'co2': safe_avg_of(environment, "ci_value"),
            'light': safe_avg_of(environment, "ir_value"),
            'hyd_temp': safe_avg_of(environment, "tl_value"),
            'hyd_ec': safe_avg_of(environment, "ei_value"),
            'hyd_ph': safe_avg_of(environment, "pl_value"),
            'stem_area': sum([box["area"] if box["id"] in stem_ids else 0 for box in data["annotations"]]),
            'leaf_area': sum([box["area"] if box["id"] in leaf_ids else 0 for box in data["annotations"]]),
        }
        
        return extracted 
    except (json.JSONDecodeError, IOError) as e:
        print(f"Error parsing {file_path}: {e}")
    except TypeError as e:
        print(f"Error parsing {file_path}: {e}")
        return None

In [81]:
parsed_records = [parse_json_file(REF_LABEL_DIR / f"{photo_path.name.split(".")[0]}.json") for photo_path in list_of_photos]

#just doing row operations here since it's way simple
for row in parsed_records:
    #copy all photos to respective id folder, easier to see
    photo_path = REF_PHOTO_DIR / f"{row['file_name']}.jpg"
    os.makedirs(REF_PHOTO_DIR / row['crops_id'], exist_ok=True)
    dst_photo_path = REF_PHOTO_DIR / row['crops_id'] / f"{row['file_name']}.jpg"
    shutil.copy(photo_path, dst_photo_path)

df = pd.DataFrame(parsed_records)

In [82]:
df.head()

,file_name,crops_id,growth,captured,kind,temp,humid,co2,light,hyd_temp,hyd_ec,hyd_ph,stem_area,leaf_area
0,C30_L01_06_003_756540,C30_L01_06_003,2,2022-01-08 02:16:24,로메인,20.0,81.0,267.5,79.0,20.70,0.05,7.4,273729.24,234047.59
1,C30_L01_06_003_806752,C30_L01_06_003,2,2022-01-14 21:16:24,로메인,19.0,83.5,262.5,59.0,19.75,0.04,7.5,544565.18,133531.10
2,C30_L01_02_002_492831,C30_L01_02_002,2,2021-09-28 23:49:07,로메인,24.0,76.5,566.0,75.0,25.30,0.14,NaN,3374691.27,2173471.15
3,C30_L01_06_001_778768,C30_L01_06_001,2,2022-01-11 16:16:24,로메인,18.0,82.5,369.5,0.0,19.90,0.05,NaN,325292.96,297204.99
4,C46_L01_02_006_571147,C46_L01_02_006,2,2021-10-22 06:16:57,로메인,18.0,52.0,493.0,18.5,18.85,0.09,6.0,1925621.20,1410172.54


In [83]:
df['kind'].unique()

<ArrowStringArray>
['로메인', '곱쓸아삭이']
Length: 2, dtype: str

In [84]:
audit_df = df.groupby('crops_id').agg(
    min_growth=('growth', 'min'),
    max_growth=('growth', 'max'),
    start_time=('captured', 'min'),
    end_time=('captured', 'max'),
    kind=('kind', lambda x: x.mode().iloc[0] if not x.mode().empty else None) #최빈값
)

growth_estimate = {
    "로메인": [0, 0, 4, 28],
    "곱쓸아삭이": [0, 0, 3, 21]
}

def calculate_t_zero(row):
    kind = row['kind']
    stage = int(row['min_growth']) # Ensure it's an int for list indexing
    
    # Get the days offset from your dict
    days_offset = growth_estimate[kind][stage]
    
    # Subtract the offset from the start_time
    return row['start_time'] - pd.Timedelta(days=days_offset)

audit_df['size'] = df.groupby('crops_id').size()
audit_df['start_time'] = pd.to_datetime(audit_df['start_time'])
audit_df['t_zero_est'] = audit_df.apply(calculate_t_zero, axis=1)
audit_df['end_time'] = pd.to_datetime(audit_df['end_time'])
audit_df['duration_days'] = (audit_df['end_time'] - audit_df['start_time']).dt.total_seconds() / 86400

# Filter for 'Perfect' Batches (Spans Stage 1 to 3)
complete_batches = audit_df[
    (audit_df['min_growth'] == 1) & (audit_df['max_growth'] == 3)
]

print(f"Total unique crops: {len(audit_df)}")
print(f"Complete (Stage 1->3) batches: {len(complete_batches)}")

Total unique crops: 39
Complete (Stage 1->3) batches: 12


In [85]:
audit_df

,min_growth,max_growth,start_time,end_time,kind,size,t_zero_est,duration_days
crops_id,,,,,,,,
C30_L01_01_001,3,3,2021-08-26 11:49:07,2021-09-07 23:49:07,로메인,41,2021-07-29 11:49:07,12.500000
C30_L01_01_002,3,3,2021-08-26 09:49:07,2021-10-06 17:16:24,로메인,289,2021-07-29 09:49:07,41.310613
C30_L01_01_003,3,3,2021-08-26 11:49:07,2021-09-01 09:49:07,로메인,102,2021-07-29 11:49:07,5.916667
C30_L01_02_001,2,3,2021-09-11 19:49:07,2021-10-08 22:16:24,로메인,179,2021-09-07 19:49:07,27.102280
C30_L01_02_002,2,3,2021-09-11 19:49:07,2021-10-08 22:16:24,로메인,302,2021-09-07 19:49:07,27.102280
C30_L01_02_003,2,3,2021-09-11 19:49:07,2021-10-08 22:16:24,로메인,140,2021-09-07 19:49:07,27.102280
C30_L01_03_001,2,3,2021-10-11 20:16:24,2021-11-05 08:16:24,로메인,218,2021-10-07 20:16:24,24.500000
C30_L01_03_002,2,3,2021-10-11 20:16:24,2021-11-05 06:16:24,로메인,226,2021-10-07 20:16:24,24.416667
C30_L01_03_003,2,3,2021-10-11 20:16:24,2021-11-05 08:16:24,로메인,224,2021-10-07 20:16:24,24.500000


In [92]:
for row in parsed_records:
    t_zero = audit_df.loc[row['crops_id'], 't_zero_est']
    row['delta_days'] = (pd.to_datetime(row['captured']) - t_zero).days

df = pd.DataFrame(parsed_records)

In [93]:
df.head()

,file_name,crops_id,growth,captured,kind,temp,humid,co2,light,hyd_temp,hyd_ec,hyd_ph,stem_area,leaf_area,delta_days
0,C30_L01_06_003_756540,C30_L01_06_003,2,2022-01-08 02:16:24,로메인,20.0,81.0,267.5,79.0,20.70,0.05,7.4,273729.24,234047.59,2
1,C30_L01_06_003_806752,C30_L01_06_003,2,2022-01-14 21:16:24,로메인,19.0,83.5,262.5,59.0,19.75,0.04,7.5,544565.18,133531.10,9
2,C30_L01_02_002_492831,C30_L01_02_002,2,2021-09-28 23:49:07,로메인,24.0,76.5,566.0,75.0,25.30,0.14,NaN,3374691.27,2173471.15,21
3,C30_L01_06_001_778768,C30_L01_06_001,2,2022-01-11 16:16:24,로메인,18.0,82.5,369.5,0.0,19.90,0.05,NaN,325292.96,297204.99,6
4,C46_L01_02_006_571147,C46_L01_02_006,2,2021-10-22 06:16:57,로메인,18.0,52.0,493.0,18.5,18.85,0.09,6.0,1925621.20,1410172.54,15


In [94]:
parsed_records_val = [parse_json_file(REF_LABEL_VAL_DIR / f"{photo_path.name.split(".")[0]}.json") for photo_path in list_of_photos_val]

for row in parsed_records_val:
    photo_path = REF_PHOTO_VAL_DIR / f"{row['file_name']}.jpg"
    os.makedirs(REF_PHOTO_VAL_DIR / row['crops_id'], exist_ok=True)
    dst_photo_path = REF_PHOTO_VAL_DIR / row['crops_id'] / f"{row['file_name']}.jpg"
    shutil.copy(photo_path, dst_photo_path)
    t_zero = audit_df.loc[row['crops_id'], 't_zero_est']
    row['delta_days'] = (pd.to_datetime(row['captured']) - t_zero).days

df_val = pd.DataFrame(parsed_records_val)

In [95]:
df_val.head()

,file_name,crops_id,growth,captured,kind,temp,humid,co2,light,hyd_temp,hyd_ec,hyd_ph,stem_area,leaf_area,delta_days
0,C30_L01_05_001_718559,C30_L01_05_001,3,2022-01-02 08:16:24,로메인,19.5,81.0,655.5,81.5,20.75,0.08,7.30,2846611.36,1615263.23,26
1,C46_L01_02_004_641843,C46_L01_02_004,1,2021-10-06 12:49:11,로메인,23.5,76.0,474.0,29.0,23.35,0.08,6.50,91519.02,77914.99,4
2,C30_L01_05_001_618571,C30_L01_05_001,2,2021-12-15 01:16:24,로메인,20.0,74.0,738.5,87.0,20.90,0.13,7.30,886693.26,686004.33,8
3,C30_L01_06_002_756672,C30_L01_06_002,2,2022-01-09 12:16:24,로메인,20.0,88.5,255.0,0.0,18.70,0.04,7.55,192004.41,160083.83,3
4,C31_L01_09_001_468379,C31_L01_09_001,2,2021-08-29 07:49:25,곱쓸아삭이,21.5,78.5,649.0,0.0,22.30,NaN,NaN,374852.21,207182.34,6


In [88]:
df_val['kind'].unique()

<ArrowStringArray>
['로메인', '곱쓸아삭이']
Length: 2, dtype: str

In [31]:
output_file = Path("plant_gotchi_train.parquet")
output_file_val = Path("plant_gotchi_validate.parquet")

# We use 'pyarrow' as the engine and 'snappy' for fast compression
df.to_parquet(
    output_file, 
    engine='pyarrow', 
    compression='snappy',
    index=False
)

print(f"Dataset(Train) successfully saved to: {output_file.absolute()}")

df_val.to_parquet(
    output_file_val, 
    engine='pyarrow', 
    compression='snappy',
    index=False
)

print(f"Dataset(Validate) successfully saved to: {output_file_val.absolute()}")

Dataset(Train) successfully saved to: /Users/mainframe/Workspace/Graduate/notebooks/plant_gotchi_train.parquet
Dataset(Validate) successfully saved to: /Users/mainframe/Workspace/Graduate/notebooks/plant_gotchi_validate.parquet
